# Install Libraries and Packages

In [2]:
%pip install python-dotenv pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


# Import Libraries

In [3]:
import os
import glob
import re
import subprocess
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

# Creation of .env file to hide security information

#### Archivo .env con las credenciales de la base de datos
#### Por favor cree este archivo en la raíz donde tiene el actual script de python con el siguiente formato:

>DB_USER=xxxx # Cambia esto por tu usuario de PostgreSQL

>DB_HOST=xxxx # Cambia esto por la dirección de tu servidor PostgreSQL (usualmente localhost)

>DB_PORT=xxxx # Cambia esto por el puerto de tu servidor PostgreSQL (usualmente 5432 o 5433)

>DB_NAME=xxxx # Cambia esto por el nombre de tu base de datos PostgreSQL

>DB_PASS=xxxx # Cambia esto por tu contraseña de PostgreSQL

# Connect Local PostgreSQL Database

In [4]:
env_path = os.path.join(os.getcwd(), '.env')
load_dotenv(dotenv_path=env_path)

USER = os.getenv("DB_USER")
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
RAW_PASS = os.getenv("DB_PASS")

if not all([USER, HOST, PORT, DB_NAME, RAW_PASS]):
    raise ValueError("Error crítico: No se pudieron importar las credenciales desde el archivo .env")

# Configure paths with raster files

In [5]:
PATH_POSTGRES_BIN = r"C:\Program Files\PostgreSQL\18\bin"
base_path = r"C:\Users\Carlos Andres\Desktop\Geovisor_Rio_Acre"

psql_abs = os.path.join(PATH_POSTGRES_BIN, "psql.exe")
raster_abs = os.path.join(PATH_POSTGRES_BIN, "raster2pgsql.exe")

if not os.path.exists(psql_abs) or not os.path.exists(raster_abs):
    raise FileNotFoundError(f"Error: No se encontraron las herramientas de PostgreSQL en: {PATH_POSTGRES_BIN}")

carpetas_principales = [
    "Conversion_HDF_GIS_Raster_p01",
    "Conversion_HDF_GIS_Raster_p02",
    "Conversion_HDF_GIS_Raster_p03",
    "Conversion_HDF_GIS_Raster_p04",
    "Conversion_HDF_GIS_Raster_p05"
]

planes_config = ["p01", "p02", "p03", "p04", "p05"]
os.environ["PGPASSWORD"] = RAW_PASS

# Insert raster in geodatabase using PostGIS

In [6]:
def extraer_fecha(nombre_archivo):
    """Parsea fechas de series de tiempo e históricos de HEC-RAS"""
    match_ts = re.search(r'(\d{2}[A-Z]{3}\d{4})_(\d{2})_(\d{2})_(\d{2})', nombre_archivo)
    if match_ts:
        try:
            str_fecha = f"{match_ts.group(1)}_{match_ts.group(2)}_{match_ts.group(3)}_{match_ts.group(4)}"
            return datetime.strptime(str_fecha, "%d%b%Y_%H_%M_%S")
        except:
            pass
            
    match_summary = re.search(r'(\d{2}[A-Z]{3}\d{4})', nombre_archivo)
    if match_summary:
        try:
            return datetime.strptime(match_summary.group(1), "%d%b%Y")
        except:
            pass
            
    return datetime(2026, 4, 19, 0, 0, 0)

def extraer_step_numero(nombre_archivo):
    """Extrae estrictamente el número de paso al inicio (ej: WSE_02_... -> 2)"""
    match = re.match(r'^WSE_(\d+)', nombre_archivo)
    if match:
        return int(match.group(1))
    return None

try:
    #===========================================================================
    # FASE 1: POBLACIÓN DE LA TABLA MAESTRA (nivel_wsel_tm) DESDE LOS EXCEL
    #===========================================================================
    print("--- INICIANDO FASE 1: CARGA DE VALORES EXCEL A TABLA MAESTRA ---")
    
    excels_en_memoria = {}
    
    for plan in planes_config:
        carpeta_excel = os.path.join(base_path, f"TimeSeries_WSEL_{plan}")
        archivo_busqueda = os.path.join(carpeta_excel, f"*_{plan}.xlsx")
        
        lista_archivos = glob.glob(archivo_busqueda)
        if not lista_archivos:
            print(f"Advertencia: No se encontró archivo Excel para el {plan} en {carpeta_excel}")
            continue
            
        ruta_excel = lista_archivos[0]
        print(f"Sincronizando registros maestros desde: {os.path.basename(ruta_excel)}")
        
        df_excel = pd.read_excel(ruta_excel)
        excels_en_memoria[plan] = df_excel
        
        for idx, row in df_excel.iterrows():
            valor_wsel = float(row['WSEL'])
            plan_str = str(row['Plan']).strip()
            
            cmd_ins_maestra = f'"{psql_abs}" -h {HOST} -U {USER} -d "{DB_NAME}" -c "INSERT INTO geovisor_data.nivel_wsel_tm (nivel_wsel, plan) VALUES ({valor_wsel:.4f}, \'{plan_str}\') ON CONFLICT (nivel_wsel, plan) DO NOTHING;"'
            subprocess.run(cmd_ins_maestra, shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
            
    print(">> Fase 1 Completada con éxito. Tabla maestra sincronizada en Postgres.\n")

    #===========================================================================
    # FASE 2: PROCESAMIENTO E INSERCIÓN RELACIONAL DE ARCHIVOS RÁSTER (SÓLO WSEL)
    #===========================================================================
    print("--- INICIANDO FASE 2: PROCESAMIENTO E IMPORTACIÓN DE RÁSTERS ---")
    
    for cp in carpetas_principales:
        sufijo_plan = cp.split('_')[-1] # Extrae 'p01', 'p02', etc.
        df_plan_excel = excels_en_memoria.get(sufijo_plan)
        
        if df_plan_excel is None:
            print(f"Saltando {cp}: No existen datos maestros en memoria para este plan.")
            continue
            
        # PROCESAMIENTO ÚNICO DE WATER SURFACE ELEVATION (WSEL)
        tabla_destino_wsel = f"geovisor_data.manchas_inundacion_wsel_{sufijo_plan}"
        subcarpetas_wsel = ["Max_WSEL_Summary", "WSE_Time_Series"]
        
        for sc in subcarpetas_wsel:
            ruta_busqueda = os.path.join(base_path, cp, sc, "*.tif")
            for ruta_tif in glob.glob(ruta_busqueda):
                nombre_archivo = os.path.basename(ruta_tif)
                fecha_timestamp = extraer_fecha(nombre_archivo).strftime("%Y-%m-%d %H:%M:%S")
                step_num = extraer_step_numero(nombre_archivo)
                
                # Sincronización directa por número de Step
                if step_num is not None and step_num <= len(df_plan_excel):
                    cota_wsel_exacta = float(df_plan_excel.iloc[step_num - 1]['WSEL'])
                else:
                    # Archivos "Summary" heredan el máximo histórico del plan como referencia
                    cota_wsel_exacta = float(df_plan_excel['WSEL'].max())
                
                # Inserción del bloque binario del Raster
                cmd_raster = f'"{raster_abs}" -a -s 32719 -f raster_wsel_{sufijo_plan} "{ruta_tif}" {tabla_destino_wsel} | "{psql_abs}" -h {HOST} -U {USER} -d "{DB_NAME}"'
                subprocess.run(cmd_raster, shell=True, check=True, stdout=subprocess.DEVNULL)
                
                # Update acoplado a las columnas específicas de la tabla de este plan
                cmd_update = f'"{psql_abs}" -h {HOST} -U {USER} -d "{DB_NAME}" -c "UPDATE {tabla_destino_wsel} SET fecha_data_{sufijo_plan} = \'{fecha_timestamp}\', nivel_wsel_{sufijo_plan} = {cota_wsel_exacta:.4f}, plan_{sufijo_plan} = \'{sufijo_plan}\' WHERE fecha_data_{sufijo_plan} IS NULL;"'
                subprocess.run(cmd_update, shell=True, check=True, stdout=subprocess.DEVNULL)
                
                print(f"Insertado WSEL [{sufijo_plan.upper()}]: {nombre_archivo} -> Cota vinculada: {cota_wsel_exacta:.4f} m")

    print("\nLas capas WSEL de los 5 planes se han consolidado perfectamente.")

except subprocess.CalledProcessError as e:
    print(f"\nError crítico en comandos nativos PostgreSQL (psql / raster2pgsql):")
    if e.stderr:
        print(e.stderr.decode('utf-8', errors='ignore'))
    else:
        print(e)
except Exception as e:
    print(f"Error General de ejecución en el pipeline de datos: {e}")

finally:
    if "PGPASSWORD" in os.environ:
        del os.environ["PGPASSWORD"]

--- INICIANDO FASE 1: CARGA DE VALORES EXCEL A TABLA MAESTRA ---
Sincronizando registros maestros desde: WSEL_Time_Series_Point_527148.5_8782230.4_p01.xlsx
Sincronizando registros maestros desde: WSEL_Time_Series_Point_527148.5_8782230.4_p02.xlsx
Sincronizando registros maestros desde: WSEL_Time_Series_Point_527148.5_8782230.4_p03.xlsx
Sincronizando registros maestros desde: WSEL_Time_Series_Point_527148.5_8782230.4_p04.xlsx
Sincronizando registros maestros desde: WSEL_Time_Series_Point_527148.5_8782230.4_p05.xlsx
>> Fase 1 Completada con éxito. Tabla maestra sincronizada en Postgres.

--- INICIANDO FASE 2: PROCESAMIENTO E IMPORTACIÓN DE RÁSTERS ---
Insertado WSEL [P01]: Summary_Max_Water_Surface_Elevation_p01.tif -> Cota vinculada: 186.5382 m
Insertado WSEL [P01]: WSE_01_19APR2026_00_00_00_p01.tif -> Cota vinculada: 178.0002 m
Insertado WSEL [P01]: WSE_02_19APR2026_00_15_00_p01.tif -> Cota vinculada: 177.6368 m
Insertado WSEL [P01]: WSE_03_19APR2026_00_30_00_p01.tif -> Cota vinculada: